# Why Cosine Similarity — Not Euclidean Distance — for NLP

---

## The Big Question

> **"We have two tools to measure how close two vectors are. Which one do we use for text — and WHY?"**

This notebook gives you a complete, proof-by-example answer. By the end you will:

1. Understand **why Euclidean distance silently fails** on text data
2. Understand **why Cosine Similarity is the right tool** for NLP
3. Build both functions **from scratch** using only NumPy
4. Prove the difference with **real text experiments**

---

### Before we start — one analogy

Imagine two book reviews of the same thriller novel:

- **Review A** (short): *"Thrilling, suspenseful, gripping plot."*
- **Review B** (long): *"Thrilling, suspenseful, gripping plot. The author builds tension masterfully. The pacing is relentless, the characters are vivid, the twists are unexpected..."* (500 more words)

Both reviews say the **same thing** — they love the same aspects. Review B just says it more times.

**Euclidean distance** sees Review B as a **giant**, far-away vector and thinks it is very different from Review A.

**Cosine similarity** ignores the length. It asks: **"Are they pointing in the same direction?"** — and correctly says: **"Yes, same opinion."**

This is the entire lesson in one analogy. The rest of this notebook proves it rigorously.

---
## Setup — Install & Import
Run this cell first. Everything else depends on it.

In [ ]:
# Install the only non-standard library we need (scikit-learn for CountVectorizer)
# NumPy, Matplotlib, and Pandas come pre-installed in Google Colab
!pip install scikit-learn matplotlib pandas numpy --quiet

In [ ]:
# ── Core numeric library — vectors, dot products, square roots ──
import numpy as np

# ── Data manipulation — display comparison tables ──
import pandas as pd

# ── Plotting — visualize vectors in 2D space ──
import matplotlib.pyplot as plt

# ── Text → Numbers — convert sentences to word-frequency vectors ──
from sklearn.feature_extraction.text import CountVectorizer

# ── Pretty output ──
pd.set_option('display.float_format', '{:.4f}'.format)
print("All libraries imported successfully!")

---
## What is a Vector? — Quick Primer for Beginners

> If you already know what a vector is in the NLP context, skip to Section 1.

A **vector** is simply **a list of numbers arranged in a fixed order**.

Think of it like a tally sheet for a sentence.

Suppose our vocabulary has 4 words: `cat`, `dog`, `meows`, `barks`

| Sentence | cat | dog | meows | barks | → Vector |
|---|---|---|---|---|---|
| "cat meows" | 1 | 0 | 1 | 0 | `[1, 0, 1, 0]` |
| "dog barks barks" | 0 | 1 | 0 | 2 | `[0, 1, 0, 2]` |
| "cat cat cat" | 3 | 0 | 0 | 0 | `[3, 0, 0, 0]` |

Each position in the vector = one word from the vocabulary.  
The number at that position = how many times that word appears.

This is called **Bag of Words (BoW)** — we put all the words in a bag and count them.

Now the question is: **how do we compare two of these vectors?**  
That's exactly what this notebook answers.

In [ ]:
# ── See it live: sentences → vectors ──────────────────────────────
# Let's run CountVectorizer on 3 tiny sentences to see exactly how it works

demo_sentences = [
    "cat meows",
    "dog barks barks",
    "cat cat cat",
]

demo_vec = CountVectorizer()
demo_matrix = demo_vec.fit_transform(demo_sentences).toarray()
demo_vocab  = demo_vec.get_feature_names_out()

# Display as a table so it's easy to read
df_demo = pd.DataFrame(
    demo_matrix,
    columns=demo_vocab,
    index=["\"cat meows\"", "\"dog barks barks\"", "\"cat cat cat\""]
)

print("Sentences converted to vectors:")
print()
print(df_demo)
print()
print("Each cell = word count for that sentence")
print("Zero means that word does NOT appear in that sentence")
print()
print("Raw vectors (just the numbers):")
for i, sent in enumerate(demo_sentences):
    print(f"  '{sent}' → {demo_matrix[i]}")

---
# SECTION 1 — CONCEPT
## Understanding the Two Distance Metrics

Before we touch text data, let's understand both metrics clearly using simple 2D vectors we can visualize.

### 1.1 — Euclidean Distance: The Ruler

Euclidean distance is what you'd measure with a **physical ruler** between two points on a map.

**Formula:**
$$d(A, B) = \sqrt{\sum_{i=1}^{n}(A_i - B_i)^2}$$

It measures the **straight-line gap** between the tips of two vectors.

- ✅ Works well when **magnitude (size) matters** (e.g., how heavy is this object?)
- ❌ Fails when **direction matters more than size** (e.g., what topic is this text about?)

In [ ]:
def euclidean_distance(a, b):
    """
    Calculates the straight-line (Euclidean) distance between two vectors.
    Think of it as the ruler distance between two points on a map.

    Args:
        a, b : numpy arrays of the same length
    Returns:
        A single float — smaller = more similar
    """
    # Step 1: subtract element-by-element → gives us a difference vector
    diff = a - b

    # Step 2: square each difference → makes negatives positive, penalises large gaps
    squared = diff ** 2

    # Step 3: sum all squared differences
    total = np.sum(squared)

    # Step 4: square root → brings back to original units (e.g., word counts, not word-count²)
    return np.sqrt(total)


# Quick sanity check on 2D points
p1 = np.array([0, 0])
p2 = np.array([3, 4])

result = euclidean_distance(p1, p2)
print(f"Distance between {p1} and {p2}: {result}")  # Expected: 5.0  (3-4-5 triangle)
print("Formula check: √(3² + 4²) =", np.sqrt(9 + 16))

### 1.2 — Cosine Similarity: The Compass

Cosine similarity measures the **angle** between two vectors — not how long they are, but **which direction they point**.

**Formula:**
$$\cos(\theta) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

**Breaking it down:**
- $A \cdot B$ = **dot product** — how much the two vectors "agree" on each dimension
- $\|A\|$ = **magnitude of A** — the length of vector A
- $\|B\|$ = **magnitude of B** — the length of vector B
- Dividing by the magnitudes = **normalizing away the length** — we only keep direction

**Output range:**
- `1.0` → same direction (identical meaning) ✅
- `0.0` → perpendicular (completely unrelated topics)
- `-1.0` → opposite directions (opposite meanings)

> **Key insight:** By dividing by the magnitudes, cosine similarity forces both vectors onto a unit sphere. A short review and a long review of the same book both land at the same point on that sphere — same direction = same meaning.

In [ ]:
def cosine_similarity(a, b):
    """
    Calculates the cosine of the angle between two vectors.
    Ignores vector length — only measures direction.

    Args:
        a, b : numpy arrays of the same length
    Returns:
        A float in [-1.0, 1.0] — higher = more similar in direction
    """
    # Step 1: dot product — multiply matching dimensions, then sum
    # Example: [1,2]·[3,4] = (1×3) + (2×4) = 11
    dot_product = np.dot(a, b)

    # Step 2: magnitude of a — the 'length' of vector a
    # np.dot(a, a) = sum of squares = a₁² + a₂² + ... then √
    magnitude_a = np.sqrt(np.dot(a, a))

    # Step 3: magnitude of b
    magnitude_b = np.sqrt(np.dot(b, b))

    # Step 4: Guard against zero vectors
    # A zero vector (all zeros) has NO direction — it means the text had
    # zero words from our vocabulary. Similarity is undefined → return 0.
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0

    # Step 5: divide — this is the normalization step that removes length
    # Now we're comparing DIRECTIONS only, not sizes
    return dot_product / (magnitude_a * magnitude_b)


# Quick test with identical vectors — cosine should be 1.0 (same direction)
v1 = np.array([1, 2, 3])
v2 = np.array([1, 2, 3])
print(f"Identical vectors:   cosine = {cosine_similarity(v1, v2):.4f}")  # Expected: 1.0

# Test with a SCALED version — same direction, different length
v3 = np.array([2, 4, 6])  # v1 × 2 — same direction, twice as big
print(f"Scaled vector (2×):  cosine = {cosine_similarity(v1, v3):.4f}")  # Still 1.0!

# Test with a zero vector — the guard should return 0.0 safely
v_zero = np.array([0, 0, 0])
print(f"Zero vector:         cosine = {cosine_similarity(v1, v_zero):.4f}")  # 0.0, no crash

# This is the KEY INSIGHT: scaling doesn't change cosine similarity
print()
print(">>> KEY INSIGHT: doubling the vector doesn't change cosine similarity.")
print("    This is why it works for text of different lengths!")

### 1.3 — Step-by-Step Manual Calculation

Before we use the functions, let's trace through both metrics **by hand** with a tiny example so there's no mystery.

We'll use:
- **A** = `[2, 0, 0]` — a document that says "AI" twice  
- **B** = `[4, 0, 0]` — the same document, but doubled in length

These represent:
```
vocabulary = [AI, football, cooking]
A: "AI AI"           → [2, 0, 0]
B: "AI AI AI AI"     → [4, 0, 0]   ← same meaning, just longer
```

We **expect** cosine = 1.0 (identical meaning) and euclidean = 2.0 (they're different in size).

In [ ]:
# ── Manual step-by-step calculation ───────────────────────────────
# vocab = [AI, football, cooking]
A = np.array([2, 0, 0])   # "AI AI"
B = np.array([4, 0, 0])   # "AI AI AI AI"

print("=" * 58)
print("  EUCLIDEAN DISTANCE — step by step")
print("=" * 58)
print(f"  A = {A}")
print(f"  B = {B}")
print()
print("  Step 1: Subtract each dimension (A - B)")
for i, (a_i, b_i) in enumerate(zip(A, B)):
    print(f"    dim[{i}]: {a_i} - {b_i} = {a_i - b_i}")

diffs = A - B
print()
print("  Step 2: Square each difference")
for i, d in enumerate(diffs):
    print(f"    dim[{i}]: ({d})² = {d**2}")

print()
total_sq = int(np.sum(diffs**2))
print(f"  Step 3: Sum all squares = {' + '.join(str(int(d**2)) for d in diffs)} = {total_sq}")
euc = euclidean_distance(A, B)
print(f"  Step 4: √{total_sq} = {euc:.4f}")
print(f"\n  ✅ Euclidean(A, B) = {euc:.4f}")
print(f"     → Says A and B are {euc:.1f} units apart (because B has more words)")

print()
print("=" * 58)
print("  COSINE SIMILARITY — step by step")
print("=" * 58)
print(f"  A = {A}")
print(f"  B = {B}")
print()

print("  Step 1: Dot product  (A · B)")
products = [f"{int(a)}×{int(b)}" for a, b in zip(A, B)]
dot_terms = [int(a)*int(b) for a, b in zip(A, B)]
print(f"    = {' + '.join(products)}")
print(f"    = {' + '.join(str(t) for t in dot_terms)} = {sum(dot_terms)}")
dot = int(np.dot(A, B))

print()
mag_A = np.sqrt(np.dot(A, A))
mag_A_sq = int(np.dot(A, A))
print(f"  Step 2: Magnitude of A")
print(f"    = √(2² + 0² + 0²) = √{mag_A_sq} = {mag_A:.4f}")

mag_B = np.sqrt(np.dot(B, B))
mag_B_sq = int(np.dot(B, B))
print(f"  Step 3: Magnitude of B")
print(f"    = √(4² + 0² + 0²) = √{mag_B_sq} = {mag_B:.4f}")

print()
print(f"  Step 4: cosine = dot ÷ (|A| × |B|)")
print(f"        = {dot} ÷ ({mag_A:.4f} × {mag_B:.4f})")
print(f"        = {dot} ÷ {mag_A * mag_B:.4f}")
cos = cosine_similarity(A, B)
print(f"        = {cos:.4f}")
print(f"\n  ✅ Cosine(A, B) = {cos:.4f}")
print(f"     → Says A and B point in the SAME direction (both are about AI)")

print()
print("=" * 58)
print("  CONCLUSION for this example:")
print(f"  Euclidean = {euc:.4f}  → 'These are different!'  ❌ (wrong for NLP)")
print(f"  Cosine    = {cos:.4f}  → 'These are identical!'  ✅ (correct for NLP)")
print()
print("  The scale factor cancels in cosine (4/2 appears in both numerator")
print("  and denominator), so the length difference disappears.")

### 1.3 — Visual Proof: Direction vs Distance

Let's plot the difference between the two metrics so we can see it clearly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Euclidean Distance vs Cosine Similarity", fontsize=16, fontweight='bold')

# ─── LEFT PLOT: Euclidean Distance ──────────────────────────────────
ax1 = axes[0]
ax1.set_title("Euclidean Distance\n(measures the gap between tips)", fontsize=12)

# Three vectors: A and B point in the same direction, C points elsewhere
# A is short, B is long — same direction as A
A = np.array([1, 1])   # short
B = np.array([4, 4])   # long — same direction as A (scaled)
C = np.array([4, 1])   # different direction, similar length to B

origin = np.array([0, 0])

# Plot the vectors as arrows from origin
ax1.annotate('', xy=A, xytext=origin, arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax1.annotate('', xy=B, xytext=origin, arrowprops=dict(arrowstyle='->', color='blue', lw=2, linestyle='dashed'))
ax1.annotate('', xy=C, xytext=origin, arrowprops=dict(arrowstyle='->', color='red', lw=2))

# Label the vectors
ax1.text(A[0]+0.1, A[1]+0.1, 'A (short review)', fontsize=10, color='blue')
ax1.text(B[0]+0.1, B[1]+0.1, 'B (long review,\nsame opinion)', fontsize=10, color='blue')
ax1.text(C[0]+0.1, C[1]+0.1, 'C (different topic)', fontsize=10, color='red')

# Show euclidean distances as dotted lines
d_AB = euclidean_distance(A, B)
d_AC = euclidean_distance(A, C)
ax1.plot([A[0], B[0]], [A[1], B[1]], 'b--', alpha=0.4)
ax1.plot([A[0], C[0]], [A[1], C[1]], 'r--', alpha=0.4)
ax1.text(2.2, 2.8, f'd(A,B)={d_AB:.2f}', fontsize=9, color='blue', alpha=0.8)
ax1.text(2.2, 0.8, f'd(A,C)={d_AC:.2f}', fontsize=9, color='red', alpha=0.8)

ax1.set_xlim(-0.3, 5.5)
ax1.set_ylim(-0.3, 5.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Dimension 1')
ax1.set_ylabel('Dimension 2')
ax1.text(0.02, 0.02, f'A→B distance = {d_AB:.2f}  ← LARGE (but same direction!)\nA→C distance = {d_AC:.2f}  ← smaller (but different direction!)',
         transform=ax1.transAxes, fontsize=9, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

# ─── RIGHT PLOT: Cosine Similarity ──────────────────────────────────
ax2 = axes[1]
ax2.set_title("Cosine Similarity\n(measures the angle between vectors)", fontsize=12)

ax2.annotate('', xy=A, xytext=origin, arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax2.annotate('', xy=B, xytext=origin, arrowprops=dict(arrowstyle='->', color='blue', lw=2, linestyle='dashed'))
ax2.annotate('', xy=C, xytext=origin, arrowprops=dict(arrowstyle='->', color='red', lw=2))

ax2.text(A[0]+0.1, A[1]+0.1, 'A (short review)', fontsize=10, color='blue')
ax2.text(B[0]+0.1, B[1]+0.1, 'B (long review,\nsame opinion)', fontsize=10, color='blue')
ax2.text(C[0]+0.1, C[1]+0.1, 'C (different topic)', fontsize=10, color='red')

cos_AB = cosine_similarity(A, B)
cos_AC = cosine_similarity(A, C)

ax2.set_xlim(-0.3, 5.5)
ax2.set_ylim(-0.3, 5.5)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('Dimension 1')
ax2.set_ylabel('Dimension 2')
ax2.text(0.02, 0.02, f'A→B cosine = {cos_AB:.4f}  ← PERFECT match (same direction!)\nA→C cosine = {cos_AC:.4f}  ← lower (different direction)',
         transform=ax2.transAxes, fontsize=9, verticalalignment='bottom',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

plt.tight_layout()
plt.show()

print("="*65)
print("EUCLIDEAN says A and B are FAR APART (d=4.24) — WRONG for NLP!")
print("COSINE    says A and B are IDENTICAL (1.0000) — CORRECT for NLP!")
print("="*65)

---
# SECTION 2 — GUIDED BUILD
## Proving the Failure of Euclidean Distance on Real Text

Let's turn sentences into vectors and see both metrics in action.

We'll use **Bag of Words (BoW)** — the simplest way to turn text into numbers:
- Build a vocabulary from all words across all documents
- Each document becomes a vector where each slot = word count

Example:
```
Vocab:    [cat, dog, barks, meows, runs, ball]
Doc A:    "cat meows"    → [1, 0, 0, 1, 0, 0]
Doc B:    "dog barks"    → [0, 1, 1, 0, 0, 0]
Doc C:    "cat cat cat cat cat meows meows"  → [5, 0, 0, 2, 0, 0]
```

Doc A and Doc C are about **the same thing** — cats. But Doc C is just more verbose. Euclidean will think C is far from A. Cosine will correctly say they're the same.

In [ ]:
# ── STEP 1: Define our sentences ──────────────────────────────────
# IMPORTANT: the "long" sentences use the EXACT SAME RATIO of words as the short ones.
# "cat meows" is 1:1 ratio → long version is "cat cat cat cat cat meows meows meows meows meows" (5:5 ratio)
# This is critical: only when ratios match does cosine = 1.0 (identical direction).
# If ratios differ, cosine < 1.0 — they're pointing in slightly different directions.

sentences = [
    # Pair 1: "cat meows" — ratio cat:meows = 1:1
    "cat meows",
    "cat cat cat cat cat meows meows meows meows meows",  # 5:5 ratio — SAME direction ✓

    # Pair 2: "football goal kick" — ratio 1:1:1
    "football goal kick",
    "football football football goal goal goal kick kick kick",  # 3:3:3 — SAME direction ✓

    # Pair 3: Different topic — about cooking (actually different from both above)
    "cooking recipe ingredients",
]

# ── STEP 2: Convert to Bag-of-Words vectors ────────────────────────
# CountVectorizer builds the vocabulary and counts word occurrences
vectorizer = CountVectorizer()

# fit_transform = (1) learn vocabulary from all sentences + (2) count words in each
X = vectorizer.fit_transform(sentences).toarray()  # .toarray() converts sparse → dense

# ── STEP 3: Display the vocabulary and vectors ────────────────────
vocab = vectorizer.get_feature_names_out()  # the list of all unique words

df_bow = pd.DataFrame(X, columns=vocab,
                      index=['Short Cat', 'Long Cat', 'Short Football',
                             'Long Football', 'Cooking'])
print("Bag-of-Words representation:")
print(df_bow)
print()
print("Each row = one sentence, each column = one word from the vocabulary")
print("The numbers = how many times that word appears in that sentence")
print()
print("Notice: Short Cat  = [1 cat, 1 meow]  vs  Long Cat = [5 cats, 5 meows]")
print("        Same RATIO → same DIRECTION in vector space → cosine = 1.0")

In [ ]:
# ── STEP 4: Check the magnitudes (lengths) of each vector ─────────
# This is WHY Euclidean distance will fail — the long sentences have
# much larger magnitudes, which makes Euclidean think they're far away.

print("Vector magnitudes (lengths):")
print("─" * 45)
labels = ['Short Cat', 'Long Cat', 'Short Football', 'Long Football', 'Cooking']
for i, label in enumerate(labels):
    # np.linalg.norm = same as our √(sum of squares)
    magnitude = np.linalg.norm(X[i])
    print(f"  {label:20s}: magnitude = {magnitude:.2f}")

print()

# Compute dynamically so the message always matches the actual values
short_cat_mag = np.linalg.norm(X[0])
long_cat_mag  = np.linalg.norm(X[1])

print(f"PROBLEM: 'Long Cat' has magnitude {long_cat_mag:.1f} but 'Short Cat' has {short_cat_mag:.1f}")
print(f"         That's a {long_cat_mag/short_cat_mag:.1f}x difference in size!")
print()
print("They mean the EXACT SAME THING (cat meows), just said differently.")
print("Euclidean sees them as far apart. Cosine will correctly see them as identical.")

In [ ]:
# ── STEP 5: Compare both metrics on all pairs ─────────────────────
short_cat      = X[0]   # "cat meows"
long_cat       = X[1]   # "cat cat cat..." (same meaning, different length)
short_football = X[2]   # "football goal kick"
long_football  = X[3]   # "football football..." (same meaning, different length)
cooking        = X[4]   # "cooking recipe ingredients" (actually different)

print("="*70)
print("COMPARISON: Euclidean Distance vs Cosine Similarity")
print("="*70)
print()

pairs = [
    ("Short Cat   vs Long Cat   (SAME meaning)",  short_cat,      long_cat,       "⚠️  SAME"),
    ("Short Cat   vs Cooking    (DIFFERENT)   ",  short_cat,      cooking,        "✅ DIFF"),
    ("Long Cat    vs Cooking    (DIFFERENT)   ",  long_cat,       cooking,        "✅ DIFF"),
    ("Short Foot  vs Long Foot  (SAME meaning)",  short_football, long_football,  "⚠️  SAME"),
    ("Short Foot  vs Cooking    (DIFFERENT)   ",  short_football, cooking,        "✅ DIFF"),
]

print(f"{'Pair':<45} {'Expected':>8}  {'Euclidean':>10}  {'Cosine':>8}")
print("-"*80)

for label, v1, v2, expected in pairs:
    euc = euclidean_distance(v1, v2)
    cos = cosine_similarity(v1, v2)
    print(f"{label:<45} {expected:>8}  {euc:>10.4f}  {cos:>8.4f}")

print()
print("─"*70)
print("KEY OBSERVATIONS:")
print()
print("1. 'Short Cat vs Long Cat' — SAME meaning but:")
print(f"   Euclidean = {euclidean_distance(short_cat, long_cat):.2f} (says they're FAR APART — WRONG!)")
print(f"   Cosine    = {cosine_similarity(short_cat, long_cat):.4f} (says they're IDENTICAL — CORRECT!)")
print()
print("2. 'Long Cat vs Cooking' — DIFFERENT topics but:")
print(f"   Euclidean = {euclidean_distance(long_cat, cooking):.2f}")
print(f"   Cosine    = {cosine_similarity(long_cat, cooking):.4f}")
print("   Cosine correctly assigns 0.0 (perpendicular — no shared direction)")

### Why does scaling not affect cosine?

Let's prove it mathematically.

Say vector **A** = `[1, 0, 1]` (cat once, meows once)  
Say vector **B** = `[5, 0, 5]` (cat 5 times, meows 5 times) — **B = 5 × A**

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

$$= \frac{A \cdot (5A)}{\|A\| \times \|5A\|}$$

$$= \frac{5(A \cdot A)}{\|A\| \times 5\|A\|}$$

$$= \frac{5(A \cdot A)}{5 \|A\|^2}$$

$$= \frac{A \cdot A}{\|A\|^2} = \frac{\|A\|^2}{\|A\|^2} = 1.0$$

**The scale factor (5) cancels out completely.** Cosine is scale-invariant by design.

In [ ]:
# ── Numerical proof of scale-invariance ───────────────────────────

a = np.array([1, 0, 1, 0, 2])   # base vector

print("Scaling the same vector by different amounts:")
print("-"*50)

for scale in [1, 2, 5, 10, 100, 1000]:
    b = a * scale   # same direction, different magnitude
    cos = cosine_similarity(a, b)
    euc = euclidean_distance(a, b)
    print(f"  scale={scale:5d}  →  cosine={cos:.6f}  euclidean={euc:.2f}")

print()
print("Cosine always = 1.000000 regardless of scale — PROVEN!")
print("Euclidean grows linearly with scale — breaks for long documents")

---
# SECTION 3 — EXPERIMENT
## The Real-World NLP Failure: Document Length Bias

Now we'll test both metrics on a classic NLP scenario:  
**"Which of these Wikipedia-style articles are most similar?"**

We'll use four topics with articles of **intentionally different lengths** to expose Euclidean's flaw.

In [ ]:
# ── Create four mini-articles on different topics ─────────────────
# INTENTIONALLY: AI doc is much longer than ML doc — same topic though!
# We expect: ML ≈ AI (same topic) and Football ≈ Tennis (both sports)

doc_ml = """
Machine learning is a branch of artificial intelligence.
Algorithms learn patterns from training data.
Supervised learning uses labeled examples.
Unsupervised learning finds hidden structure in data.
Neural networks and deep learning are powerful techniques.
Models are trained to minimize prediction error.
"""

# AI document is ~3x longer than ML — but same topic!
doc_ai = """
Artificial intelligence is the simulation of human intelligence.
Machine learning is a core subset of artificial intelligence.
Deep learning uses neural networks with many layers.
AI systems learn from data to improve performance over time.
Natural language processing enables machines to understand human language.
Computer vision allows machines to interpret images and video.
Reinforcement learning trains agents through reward and penalty.
Artificial intelligence applications span healthcare diagnostics and self-driving vehicles.
Training large language models requires massive computational resources.
Generative AI creates new content including text images and code.
AI researchers study algorithms data and computing infrastructure.
Supervised unsupervised and reinforcement learning are the three main learning paradigms.
Artificial intelligence is transforming every industry in the world today.
Machine learning engineers build train and deploy intelligent systems.
Neural network architectures include transformers convolutional and recurrent designs.
"""

doc_football = """
Football is a popular team sport played worldwide.
Players kick the ball to score goals.
The World Cup is the biggest football tournament.
A match consists of two halves of forty five minutes each.
The goalkeeper defends the goal from the opposing team.
"""

doc_tennis = """
Tennis is a racket sport played on a court.
Players hit the ball over the net to score points.
Grand Slam tournaments include Wimbledon and the US Open.
A tennis match is divided into sets and games.
The serve begins each point in a tennis game.
"""

docs = [doc_ml, doc_ai, doc_football, doc_tennis]
doc_names = ["Machine Learning", "Artificial Intelligence", "Football", "Tennis"]

# Check document lengths — this is what creates the bias
print("Document word counts:")
for name, doc in zip(doc_names, docs):
    print(f"  {name:25s}: {len(doc.split()):3d} words")

print()
print("Notice: AI doc is much longer than ML doc — but same topic!")
print("Euclidean will incorrectly see them as different because of length.")

In [ ]:
# ── Vectorize all four documents ──────────────────────────────────

# Create a new vectorizer fitted on our four documents
doc_vectorizer = CountVectorizer(stop_words='english')  # remove 'the', 'is', 'a', etc.

# Transform all 4 documents into word-count vectors
X_docs = doc_vectorizer.fit_transform(docs).toarray()

print(f"Vocabulary size: {len(doc_vectorizer.vocabulary_)} unique words")
print(f"Each document is now a vector of length {X_docs.shape[1]}")
print()

# Check the raw vector magnitudes (lengths)
print("Vector magnitudes after vectorization:")
for i, name in enumerate(doc_names):
    mag = np.linalg.norm(X_docs[i])
    print(f"  {name:25s}: magnitude = {mag:.2f}")

print()
print("AI has a much larger magnitude than ML — this will fool Euclidean!")

In [ ]:
# ── TEST 1: Euclidean Distance ────────────────────────────────────
# Reference document: Machine Learning
# Question: Which of the others is most similar to ML?

ml_vec = X_docs[0]   # Machine Learning vector
ai_vec = X_docs[1]   # Artificial Intelligence vector
fb_vec = X_docs[2]   # Football vector
tn_vec = X_docs[3]   # Tennis vector

print("EUCLIDEAN DISTANCE from 'Machine Learning' to all others:")
print("-"*55)
print("(SMALLER = more similar)")
print()

d_ml_ai = euclidean_distance(ml_vec, ai_vec)
d_ml_fb = euclidean_distance(ml_vec, fb_vec)
d_ml_tn = euclidean_distance(ml_vec, tn_vec)

print(f"  ML → AI (same topic!)  : {d_ml_ai:.4f}")
print(f"  ML → Football          : {d_ml_fb:.4f}")
print(f"  ML → Tennis            : {d_ml_tn:.4f}")

# Find what Euclidean thinks is closest
distances = {'AI': d_ml_ai, 'Football': d_ml_fb, 'Tennis': d_ml_tn}
closest_euc = min(distances, key=distances.get)

print()
print(f"  Euclidean says ML is closest to: >>> {closest_euc} <<<")
print()
print("  Expected: ML closest to AI (same field!)")

if closest_euc != 'AI':
    print(f"  RESULT: WRONG! Euclidean is fooled by document length bias!")
else:
    print(f"  RESULT: Correct (but this depends on our specific doc lengths)")

In [ ]:
# ── TEST 2: Cosine Similarity ─────────────────────────────────────

print("COSINE SIMILARITY from 'Machine Learning' to all others:")
print("-"*55)
print("(HIGHER = more similar, max = 1.0)")
print()

c_ml_ai = cosine_similarity(ml_vec, ai_vec)
c_ml_fb = cosine_similarity(ml_vec, fb_vec)
c_ml_tn = cosine_similarity(ml_vec, tn_vec)

print(f"  ML → AI (same topic!)  : {c_ml_ai:.4f}")
print(f"  ML → Football          : {c_ml_fb:.4f}")
print(f"  ML → Tennis            : {c_ml_tn:.4f}")

cosines = {'AI': c_ml_ai, 'Football': c_ml_fb, 'Tennis': c_ml_tn}
closest_cos = max(cosines, key=cosines.get)

print()
print(f"  Cosine says ML is closest to: >>> {closest_cos} <<<")
print()
if closest_cos == 'AI':
    print("  RESULT: CORRECT! Cosine correctly identifies AI as most similar to ML")
    print("  (ignores the fact that the AI doc is much longer)")

In [ ]:
# ── Side-by-side visualization of the full similarity matrix ──────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Full Similarity Matrix: Euclidean vs Cosine", fontsize=14, fontweight='bold')

n = len(doc_names)

# Build Euclidean distance matrix
euc_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        euc_matrix[i, j] = euclidean_distance(X_docs[i], X_docs[j])

# Build Cosine similarity matrix
cos_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        cos_matrix[i, j] = cosine_similarity(X_docs[i], X_docs[j])

# ─── Heatmap helper ──────────────────────────────────────────────
def plot_heatmap(ax, matrix, title, cmap, fmt):
    im = ax.imshow(matrix, cmap=cmap, aspect='auto')
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(doc_names, rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(doc_names, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, fmt.format(matrix[i, j]),
                    ha='center', va='center', fontsize=9,
                    color='white' if matrix[i, j] > matrix.max() * 0.6 else 'black')

plot_heatmap(axes[0], euc_matrix,
             "Euclidean Distance\n(smaller = more similar, 0 = identical)",
             cmap='Reds_r', fmt='{:.1f}')

plot_heatmap(axes[1], cos_matrix,
             "Cosine Similarity\n(larger = more similar, 1 = identical)",
             cmap='Blues', fmt='{:.3f}')

plt.tight_layout()
plt.show()

print()
print("READ THE HEATMAPS:")
print("Cosine (right): ML and AI have HIGH similarity (bright blue) — CORRECT")
print("                Football and Tennis also HIGH (both sports) — CORRECT")
print("Euclidean (left): ML and AI may NOT appear closest — LENGTH DISTORTION")

### The Fix: L2 Normalization makes Euclidean work like Cosine

Here's an important insight: **if you L2-normalize your vectors first, Euclidean distance becomes equivalent to Cosine similarity.**

This is because after normalization all vectors have magnitude = 1, so the length bias disappears.

In practice, NLP uses Cosine similarity directly (no manual normalization step needed).

In [ ]:
def l2_normalize(v):
    """Normalize a vector to unit length (magnitude = 1)."""
    # Divide every element by the vector's magnitude
    return v / np.sqrt(np.dot(v, v))


# Normalize all document vectors
X_normalized = np.array([l2_normalize(X_docs[i]) for i in range(n)])

print("After L2 normalization — all magnitudes should be 1.0:")
for i, name in enumerate(doc_names):
    mag = np.linalg.norm(X_normalized[i])
    print(f"  {name:25s}: magnitude = {mag:.6f}")

print()
print("Euclidean on NORMALIZED vectors from ML to others:")
print("─"*50)

for i, name in enumerate(doc_names[1:], 1):  # skip ML itself
    euc_norm = euclidean_distance(X_normalized[0], X_normalized[i])
    cos_val  = cosine_similarity(X_docs[0], X_docs[i])

    # Mathematical relationship: euclidean(normalized) = sqrt(2*(1 - cosine))
    expected = np.sqrt(2 * (1 - cos_val))

    print(f"  ML → {name:25s}: euc_normalized={euc_norm:.4f}  cosine={cos_val:.4f}  √(2(1-cos))={expected:.4f}")

print()
print("Formula: euclidean(normalized_A, normalized_B) = √(2 × (1 - cosine(A, B)))")
print("They carry exactly the same information when vectors are normalized.")

---
# SECTION 4 — CHALLENGE
## Test Your Understanding

Try these exercises to solidify what you've learned.

### Challenge 1 — Tweet Classifier

We have a reference knowledge base of three topics. Given a new tweet, use cosine similarity to automatically classify which topic it belongs to.

**Your task:** Complete the `classify_tweet()` function.

In [ ]:
# ── Knowledge base of reference documents ────────────────────────
knowledge_base = {
    "Tech / AI"     : "artificial intelligence machine learning neural network algorithm data model training",
    "Sports"        : "goal match player team score league championship tournament athlete competition",
    "Food & Cooking": "recipe ingredients cook bake chef kitchen meal flavor cuisine dish restaurant",
}

# ── Vectorize the knowledge base ──────────────────────────────────
kb_vectorizer = CountVectorizer()
kb_docs   = list(knowledge_base.values())
kb_labels = list(knowledge_base.keys())
KB_vectors = kb_vectorizer.fit_transform(kb_docs).toarray()

print(f"Knowledge base vocabulary: {list(kb_vectorizer.get_feature_names_out())}")
print()


def classify_tweet(tweet: str):
    """
    Classify a tweet into one of the knowledge base categories
    using cosine similarity.

    Args:
        tweet: the input tweet as a plain string
    Returns:
        (label, score) — the best matching category and its cosine score
    """
    # TODO 1: Vectorize the tweet using kb_vectorizer
    # IMPORTANT: use .transform() NOT .fit_transform()
    # → .fit_transform() would rebuild the vocabulary (losing our KB vocab!)
    # → .transform() maps the tweet into the EXISTING vocabulary
    # Hint: kb_vectorizer.transform([tweet]).toarray()[0]
    tweet_vec = ___

    # TODO 2: Calculate cosine similarity between tweet_vec and each KB vector
    # Hint: loop over enumerate(KB_vectors), build a list of (similarity, label) tuples
    scores = []
    for i, kb_vec in enumerate(KB_vectors):
        sim = ___  # cosine_similarity(tweet_vec, kb_vec)
        scores.append((sim, kb_labels[i]))

    # TODO 3: Return the label with the HIGHEST similarity score
    # Hint: max(scores) picks the tuple with the largest first element (the score)
    best_score, best_label = ___
    return best_label, best_score


# ── Test tweets (don't change these) ─────────────────────────────
# Note: tweets must contain words that exist in our KB vocabulary.
# Tweets with NO shared words produce a zero vector (similarity = 0).
test_tweets = [
    "Machine learning model achieves new algorithm benchmark",       # → Tech/AI
    "What a match! The championship goal in extra time was incredible",  # → Sports
    "Just made the best recipe with fresh ingredients from the market",  # → Food
    "Training a neural network to classify data",                    # → Tech/AI
]

print("Tweet Classification Results:")
print("─" * 60)
for tweet in test_tweets:
    label, score = classify_tweet(tweet)
    print(f"  Tweet: '{tweet[:50]}'")
    print(f"  → Category: {label}  (cosine = {score:.4f})")
    print()

#### Challenge 1 — Model Answer
> Try it yourself first. Only scroll here once you've attempted the TODOs above.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CHALLENGE 1 — MODEL ANSWER                              ║
# ║  Try it yourself FIRST. Only run this after attempting.  ║
# ╚══════════════════════════════════════════════════════════╝

def classify_tweet_solution(tweet: str):
    # TODO 1 SOLUTION: transform the tweet into the existing vocabulary
    # transform() maps new text into the vocabulary we already built
    # [0] extracts the first (and only) row from the 2D array
    tweet_vec = kb_vectorizer.transform([tweet]).toarray()[0]

    # TODO 2 SOLUTION: compare against each category
    scores = []
    for i, kb_vec in enumerate(KB_vectors):
        sim = cosine_similarity(tweet_vec, kb_vec)  # ← fill in the function
        scores.append((sim, kb_labels[i]))

    # TODO 3 SOLUTION: pick the highest score
    # max() on a list of tuples compares by the first element (sim)
    best_score, best_label = max(scores)
    return best_label, best_score


print("=== MODEL ANSWER: Tweet Classifier ===")
print()
for tweet in test_tweets:
    label, score = classify_tweet_solution(tweet)
    print(f"  '{tweet[:50]}'")
    print(f"  → {label}  (cosine = {score:.4f})")
    print()

print("KEY POINT: We use .transform() not .fit_transform()")
print("  .fit_transform() = learn vocabulary + vectorize (for training data)")
print("  .transform()     = vectorize only using the EXISTING vocabulary (for new data)")

### Challenge 2 — Duplicate Tweet Detector

Build a function that detects near-duplicate tweets.

A tweet is a "near-duplicate" if its cosine similarity with any existing tweet exceeds a threshold.

**Your task:** Complete the `find_duplicates()` function.

In [ ]:
tweets = [
    "Machine learning is amazing technology",                       # 0
    "Machine learning is an absolutely amazing technology",         # 1 — near dup of 0 (same words +1)
    "I love watching football and soccer games",                    # 2
    "I love playing football and soccer games",                     # 3 — near dup of 2 (watching→playing)
    "Deep learning neural networks achieve great results",          # 4 — distinct tech tweet
    "Cooking pasta with fresh tomatoes and basil is delicious",     # 5 — completely different topic
    "Machine learning is a powerful and amazing technology",        # 6 — near dup of 0
]


def find_duplicates(tweets: list, threshold: float = 0.7):
    """
    Find all pairs of tweets with cosine similarity above the threshold.

    Args:
        tweets    : list of tweet strings
        threshold : cosine similarity cutoff (default 0.7 = 70% similar)
    Returns:
        List of tuples: (index_i, index_j, similarity_score)
    """
    # Step 1: vectorize all tweets in one go
    dup_vec = CountVectorizer()
    T = dup_vec.fit_transform(tweets).toarray()

    duplicates = []

    # Step 2: compare every pair (i, j) where i < j
    # This avoids comparing (0,1) AND (1,0) — they're the same pair
    # It also avoids comparing a tweet with itself (always 1.0)
    for i in range(len(tweets)):
        for j in range(i + 1, len(tweets)):   # start from i+1, not 0

            # TODO: calculate cosine similarity between tweet i and tweet j
            sim = ___

            # TODO: if similarity >= threshold, record this pair
            if ___:
                duplicates.append((i, j, sim))

    return duplicates


# Run the detector at threshold=0.7
found = find_duplicates(tweets, threshold=0.7)

print("Near-Duplicate Pairs Found (threshold=0.7):")
print("─" * 70)
for i, j, sim in found:
    print(f"  [{i}] '{tweets[i]}'")
    print(f"  [{j}] '{tweets[j]}'")
    print(f"       Cosine similarity: {sim:.4f}")
    print()

#### Challenge 2 — Model Answer
> Try it yourself first. Only scroll here once you've attempted the TODOs above.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CHALLENGE 2 — MODEL ANSWER                              ║
# ╚══════════════════════════════════════════════════════════╝

def find_duplicates_solution(tweets: list, threshold: float = 0.7):
    dup_vec = CountVectorizer()
    T = dup_vec.fit_transform(tweets).toarray()

    duplicates = []
    for i in range(len(tweets)):
        for j in range(i + 1, len(tweets)):
            # TODO SOLUTION: cosine between tweet i and tweet j
            sim = cosine_similarity(T[i], T[j])

            # TODO SOLUTION: keep if above threshold
            if sim >= threshold:
                duplicates.append((i, j, sim))

    return duplicates


print("=== MODEL ANSWER: Duplicate Tweet Detector ===")
print()
found = find_duplicates_solution(tweets, threshold=0.7)

print(f"Found {len(found)} duplicate pairs at threshold=0.7:")
print()
for i, j, sim in found:
    print(f"  [{i}] '{tweets[i]}'")
    print(f"  [{j}] '{tweets[j]}'")
    print(f"       Cosine = {sim:.4f}")
    print()

print("─" * 60)
print("WHY i < j in the inner loop:")
print("  range(i+1, len) ensures:")
print("    (a) No self-comparison  → tweet vs itself always = 1.0")
print("    (b) No reverse duplicates → pair (0,1) and (1,0) are the same")
print()
print("NOTICE tweets [0], [1], [6] all cluster together — they're all")
print("variations of the same 'Machine learning is amazing' sentence.")

### Challenge 3 — When DOES Euclidean Work Better?

Cosine similarity is not always the right choice. Think about these scenarios and write your reasoning:

1. You're comparing animal body measurements (height, weight, wingspan). Which metric?
2. You're building a book recommendation system. Which metric?
3. You're detecting anomalies in sensor readings (temperature, pressure, vibration). Which metric?
4. You're matching job descriptions to resumes. Which metric?

Then run the cell below to see the demonstration for scenario 1.

In [ ]:
# ── Scenario 1: Animal measurements — Euclidean wins ─────────────
# Magnitude (actual size) matters here!
# A mouse and a cat are biologically very different — even if their
# weight:height:wingspan ratio is similar.

# [weight_kg, height_cm, leg_length_cm]
mouse  = np.array([0.02, 7,   1.5])   # tiny
cat    = np.array([4.0,  25,  15.0])  # medium
lion   = np.array([190,  120, 80.0])  # large

# TODO: try both metrics and decide which makes more biological sense
print("Mouse vs Cat:")
print(f"  Euclidean: {euclidean_distance(mouse, cat):.2f}")
print(f"  Cosine:    {cosine_similarity(mouse, cat):.4f}")
print()
print("Mouse vs Lion:")
print(f"  Euclidean: {euclidean_distance(mouse, lion):.2f}")
print(f"  Cosine:    {cosine_similarity(mouse, lion):.4f}")
print()
print("Cat vs Lion:")
print(f"  Euclidean: {euclidean_distance(cat, lion):.2f}")
print(f"  Cosine:    {cosine_similarity(cat, lion):.4f}")
print()
print("Your analysis:")
print("  Cosine: mouse and cat have HIGH similarity (both have similar proportions)")
print("  Euclidean: mouse and cat are FAR apart (size matters for biology!)")
print()
print("For body measurements → Euclidean is CORRECT.")
print("For text content → Cosine is CORRECT.")

---
# Summary — When to Use Each Metric

| Situation | Use | Why |
|---|---|---|
| **Text similarity** (BoW, TF-IDF, embeddings) | Cosine | Document length doesn't indicate topic relevance |
| **Short vs long version of same text** | Cosine | Scale-invariant by design |
| **Search engines, recommendation** | Cosine | Topic direction matters, not word count |
| **Physical measurements** (height, weight) | Euclidean | Magnitude carries real information |
| **Anomaly detection in sensor data** | Euclidean | Size of deviation matters |
| **Image pixels** | Euclidean (or Cosine, context-dependent) | Depends whether brightness matters |
| **Normalized vectors** (magnitude = 1) | Either | They become mathematically equivalent |

---

## The One-Line Rule

> **If the SIZE of your vectors doesn't carry meaningful information, use Cosine Similarity.**  
> In NLP, a longer document is not a "bigger" topic — it's just more verbose. So we use Cosine.

---

## What's MISSING? →

We're measuring similarity between texts using **word counts**. But word counts have a problem:  
- Common words like *"the", "is", "a"* appear everywhere and distort similarity.
- The word *"Python"* appearing once in a short doc is more informative than appearing 10 times in a 10,000-word doc.

**Next:** TF-IDF — a smarter weighting scheme that fixes exactly this problem.